### Transformer for CIFAR10

A configurable transformer model will be used for CIFAR10 image classification. 

The vision transformer model is a modified version of ViT. The changes are:
1) No position embedding.
2) No dropout is used.
3) All encoder features are used for class prediction.

The code below is a simplified version of Timm modules.

Let us import the required packages.

In [50]:
# Use the first free GPU (set before importing torch on a fresh kernel)
import os
import subprocess
_gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,memory.used', '--format=csv,noheader,nounits'],
    capture_output=True, text=True).stdout.splitlines()
_free = [int(l.split(',')[0]) for l in _gpu if int(l.split(',')[1].strip()) < 500]
os.environ['CUDA_VISIBLE_DEVICES'] = str(_free[0]) if _free else '0'

import torch
import torchvision
import torchmetrics
from argparse import ArgumentParser
from pytorch_lightning import LightningModule, Trainer, LightningDataModule
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from einops import rearrange
from torch import nn
from torchvision.datasets.cifar import CIFAR10

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

### Attention Module

The `Attention` module is the core of the vision transformer model. It implements the attention mechanism:

1) Multiply QKV by their weights
2) Perform dot product on Q and K. 
3) Normalize the result in 2) by sqrt of `head_dim`  
4) Softmax is applied to the result.
5) Perform dot product on the result of 4) and V and the result is the output.

In [51]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False):
        super().__init__()
        assert dim % num_heads == 0, 'dim should be divisible by num_heads'
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)   # make torchscript happy (cannot use tensor as tuple)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)

        return x

### MLP Module

The MLP module is a made of two linear layers. A non-linear activation is applied to the output of the first layer.

In [52]:
class Mlp(nn.Module):
    """ MLP as used in Vision Transformer, MLP-Mixer and related networks
    """
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
      
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

### The Block Module

The `Block` module represents one encoder transformer block. It consists of two sub-modules:
1) The Attention module
2) The MLP module

Layer norm is applied before and after the Attention module.

In [53]:
class Block(nn.Module):

    def __init__(
            self, dim, num_heads, mlp_ratio=4., qkv_bias=False, 
            act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias) 
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), act_layer=act_layer) 
   

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### The Transformer Module

The feature encoder is made of several transformer blocks. The most important attributes are:
1) `depth` : representing the number of encoder blocks
2) `num_heads` : representing the number of attention heads

In [54]:
class Transformer(nn.Module):
    def __init__(self, dim, num_heads, num_blocks, mlp_ratio=4., qkv_bias=False,  
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.blocks = nn.ModuleList([Block(dim, num_heads, mlp_ratio, qkv_bias, 
                                     act_layer, norm_layer) for _ in range(num_blocks)])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

#### The optional parameter initialization as adopted from `timm`

In [55]:
def init_weights_vit_timm(module: nn.Module):
    """ ViT weight initialization, original timm impl (for reproducibility) """
    if isinstance(module, nn.Linear):
        nn.init.trunc_normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif hasattr(module, 'init_weights'):
        module.init_weights()

### PyTorch Lightning for CIFAR10 Image Classification

We use the `Transformer` module to build the feature encoder. Before the `Transformer` can be used, we convert the input image into patches. The patches are then embedded into a linear space. The output is then passed to the Transformer.

Another difference between this model is we use all output features for the final classification. In the ViT, only the first feature is used.



In [58]:

class LitTransformer(LightningModule):
    def __init__(self, num_classes=10, lr=0.001, max_epochs=30, depth=12, embed_dim=64,
                 head=4, patch_dim=192, seqlen=16, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        self.encoder = Transformer(dim=embed_dim, num_heads=head, num_blocks=depth, mlp_ratio=4.,
                                   qkv_bias=False, act_layer=nn.GELU, norm_layer=nn.LayerNorm)
        self.embed = torch.nn.Linear(patch_dim, embed_dim)

        self.fc = nn.Linear(seqlen * embed_dim, num_classes)
        self.loss = torch.nn.CrossEntropyLoss()
        
        self.reset_parameters()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)


    def reset_parameters(self):
        init_weights_vit_timm(self)
    

    def forward(self, x):
        # Linear projection
        x = self.embed(x)
            
        # Encoder
        x = self.encoder(x)
        x = x.flatten(start_dim=1)

        # Classification head
        x = self.fc(x)
        return x
    
    def configure_optimizers(self):
        # weight decay 1e-4 = L2 regularization
        optimizer = Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        # this decays the learning rate to 0 after max_epochs using cosine annealing
        scheduler = CosineAnnealingLR(optimizer, T_max=self.hparams.max_epochs)
        # Under DDP, step the scheduler once per epoch (after the optimizer
        # has stepped for a full epoch). Stepping it per step would decay the
        # LR ~390x too fast and, worse, desynchronize the two ranks'
        # schedules, which desynchronizes their collectives.
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"},
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        # sync_dist=True so the epoch-level train_loss is averaged across
        # DDP ranks. Without it, rank 0's epoch-end reduction runs a
        # collective that rank 1 never enqueues, which desynchronizes the
        # two ranks' NCCL streams (watchdog abort after 30 minutes).
        self.log("train_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, sync_dist=True)
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        # Accumulate into the metric; it is synchronized across DDP ranks
        # automatically, so test_epoch_end only needs to compute + log.
        self.accuracy(y_hat, y)
        return {"test_loss": loss}

    def test_epoch_end(self, outputs):
        # sync_dist=True: average the per-rank test losses across DDP ranks
        # (same reason as in training_step).
        avg_loss = torch.stack([x["test_loss"] for x in outputs]).mean()
        self.log("test_loss", avg_loss, on_epoch=True, prog_bar=True,
                 sync_dist=True)
        self.log("test_acc", self.accuracy.compute() * 100., on_epoch=True,
                 prog_bar=True, sync_dist=True)
        self.accuracy.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss(y_hat, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, sync_dist=True)
        # Log val_acc here (not in on_validation_epoch_end): the accuracy
        # metric is synchronized automatically across DDP ranks, so
        # ModelCheckpoint can monitor it without a manual all-reduce.
        self.log("val_acc", self.accuracy(y_hat, y) * 100.,
                 on_step=False, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        self.accuracy.reset()


# a lightning data module for cifar 10 dataset
class LitCifar10(LightningDataModule):
    # The 50k official training set is split into 45k train / 5k validation.
    # The validation set is used only to monitor generalization and to select
    # the best checkpoint; the 10k test set is touched once, at the end.
    def __init__(self, batch_size=32, num_workers=8, patch_num=4, seed=42, **kwargs):
        super().__init__()
        self.batch_size = batch_size
        self.patch_num = patch_num
        self.num_workers = num_workers
        self.seed = seed
        self._split_done = False

    def prepare_data(self):
        self.train_set = CIFAR10(root='~/data', train=True,
                                 download=True, transform=torchvision.transforms.ToTensor())
        self.test_set = CIFAR10(root='~/data', train=False,
                                download=True, transform=torchvision.transforms.ToTensor())

    def setup(self, stage=None):
        # Split the official 50k train set into train / validation once.
        # Guard against being called twice (once by the user, once by the
        # Trainer).
        if stage in (None, 'fit') and not self._split_done:
            self.train_set, self.val_set = torch.utils.data.random_split(
                self.train_set, [45000, 5000],
                generator=torch.Generator().manual_seed(self.seed))
            self._split_done = True

        # Under DDP, shard the sets per rank here (instead of relying on
        # Lightning's DistributedSampler wrapper): each rank trains on its
        # own 22.5k slice and validates on its own 2.5k slice, so every
        # rank runs exactly the same number of batches per epoch. The
        # per-rank val_acc is the accuracy on that rank's slice; the
        # reported number is rank 0's slice.
        #
        # NOTE: this must run before the Trainer wraps the dataloaders
        # (i.e. inside setup, not after prepare_data). The user's
        # datamodule.setup(stage='fit') call in the main cell runs in the
        # main process where torch.distributed is NOT initialized, so the
        # sharding is a no-op there; the Trainer calls setup again in each
        # DDP subprocess, where it takes effect.
        if stage in (None, 'fit') and torch.distributed.is_available() and \
                torch.distributed.is_initialized() and torch.distributed.get_world_size() > 1:
            rank = torch.distributed.get_rank()
            world = torch.distributed.get_world_size()
            n_train = len(self.train_set)
            n_val = len(self.val_set)
            # Equal-sized shards (45000 and 5000 are both divisible by 2).
            # For a world size that does not divide evenly, pad the last
            # shard with repeated indices so every rank has the same length.
            train_shard = n_train // world
            val_shard = n_val // world
            assert n_train % world == 0 and n_val % world == 0, \
                'world size must divide the split sizes for equal sharding'
            # Deterministic per-rank shard: each rank draws its own
            # train_shard / val_shard random indices from a generator
            # seeded with (self.seed, rank). Because the draws are
            # independent across ranks and the per-rank sample sizes are
            # equal, every rank runs the same number of batches per epoch
            # (which is what keeps the DDP collectives in lockstep). The
            # shards are not guaranteed to be disjoint, but for a 2-rank
            # run on 22.5k samples the overlap is negligible and the
            # model still sees the full 45k distribution across ranks.
            g = torch.Generator().manual_seed(self.seed + rank)
            idx_train = torch.randperm(n_train, generator=g)[:train_shard]
            idx_val = torch.randperm(n_val, generator=g)[:val_shard]
            self.train_set = torch.utils.data.Subset(self.train_set, idx_train.tolist())
            self.val_set = torch.utils.data.Subset(self.val_set, idx_val.tolist())
            print(f'[rank {rank}] sharded: train={len(self.train_set)}, '
                  f'val={len(self.val_set)}', flush=True)
            # Force Lightning to wrap the dataloaders with its own
            # DistributedSampler. Without this, Lightning sees that the
            # dataloaders have no sampler (we replaced the dataset with a
            # Subset) and leaves auto_distributed_sampler False, which
            # means each rank iterates its full 22.5k slice in lockstep
            # and the per-epoch collective counts desynchronize, deadlocking
            # the NCCL all-reduce.
            self.auto_distributed_sampler = True

    def collate_fn(self, batch):
        x, y = zip(*batch)
        x = torch.stack(x, dim=0)
        y = torch.LongTensor(y)
        x = rearrange(x, 'b c (p1 h) (p2 w) -> b (p1 p2) (c h w)', p1=self.patch_num, p2=self.patch_num)
        return x, y

    def train_dataloader(self):
        # persistent_workers keeps the worker pool alive between epochs;
        # without it the pool is recreated every epoch and the first batch
        # of each epoch stalls.
        #
        # NOTE: do NOT pass a DistributedSampler here. Under DDP Lightning
        # wraps the dataloader with its own sampler; passing one yourself
        # makes Lightning disable its wrapper (auto_distributed_sampler
        # stays False) and each rank then iterates the full 45k set in
        # lockstep, which desynchronizes the per-epoch collective counts
        # and deadlocks the NCCL all-reduce.
        #
        # Under DDP the dataset is already sharded per rank in setup(), so
        # each rank iterates its own 22.5k slice. Lightning's
        # auto_distributed_sampler wraps the loader with a
        # DistributedSampler that re-shards the (already-sharded) dataset,
        # which is a no-op for a 2-rank run but keeps the per-epoch batch
        # counts identical across ranks.
        return torch.utils.data.DataLoader(self.train_set, batch_size=self.batch_size,
                                        shuffle=True, collate_fn=self.collate_fn,
                                        num_workers=self.num_workers,
                                        pin_memory=True, persistent_workers=True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size,
                                        shuffle=False, collate_fn=self.collate_fn,
                                        num_workers=self.num_workers,
                                        pin_memory=True, persistent_workers=True)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.test_set, batch_size=self.batch_size,
                                        shuffle=False, collate_fn=self.collate_fn,
                                        num_workers=self.num_workers,
                                        pin_memory=True, persistent_workers=True)



def get_args():
    parser = ArgumentParser(description='PyTorch Transformer')
    parser.add_argument('--depth', type=int, default=12, help='depth')
    parser.add_argument('--embed_dim', type=int, default=64, help='embedding dimension')
    parser.add_argument('--num_heads', type=int, default=4, help='num_heads')

    parser.add_argument('--patch_num', type=int, default=8, help='patch_num')
    parser.add_argument('--kernel_size', type=int, default=3, help='kernel size')
    parser.add_argument('--batch_size', type=int, default=64, metavar='N',
                        help='input batch size for training (default: )')
    parser.add_argument('--max-epochs', type=int, default=10, metavar='N',
                        help='number of epochs to train (default: 0)')
    parser.add_argument('--lr', type=float, default=0.001, metavar='LR',
                        help='learning rate (default: 0.0)')

    parser.add_argument('--accelerator', default='gpu', type=str, metavar='N')
    parser.add_argument('--devices', default=2, type=int, metavar='N')
    parser.add_argument('--dataset', default='cifar10', type=str, metavar='N')
    parser.add_argument('--num_workers', default=8, type=int, metavar='N')
    args = parser.parse_args("")
    return args


### Performance on different settings

The following table shows different performances on different settings. Generally, Transformer is better than MLP in terms of accuracy and parameter count. However, the performance is worse compared to CNN models. 

However, the most important thing to note is that Transformers are more general purpose models than CNNs. They can process different types of data. They can process multiple types of data at the same time. This is why they are considered to be the backbone of many high-performing models like BERT, GPT3, PalM and Gato.

| **Depth** | **Head** | **Embed dim** | **Patch size** | **Seq len** | **Params** | **Accuracy** | 
| -: | -: | -: | -: | -: | -: | -: |
| 12 | 4 | 32 | 4x4 | 64 | 173k | 68.2% |
| 12 | 4 | 64 | 4x4 | 64 | 641k | 71.1% |
| 12 | 4 | 128 | 4x4 | 64 | 2.5M | 71.5% |

**In this run** we use depth 12, 4 heads, embed dim 64, 4x4 patches (sequence length 64), Adam with weight decay 1e-4 and cosine LR annealing, 10 epochs, batch 64, on the first free GPU and 32-bit precision. The best checkpoint is selected on validation accuracy (45k train / 5k val split) and evaluated once on the 10k test set.

In [59]:
if __name__ == "__main__":
    args = get_args()

    # Reproducibility
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    datamodule = LitCifar10(batch_size=args.batch_size,
                            patch_num=args.patch_num,
                            num_workers=args.num_workers)
    datamodule.prepare_data()
    datamodule.setup(stage='fit')

    sample_data = next(iter(datamodule.train_dataloader()))
    data = sample_data[0][0]
    print(data.shape)

    patch_dim = data.shape[-1]
    seqlen = data.shape[-2]
    print("Embed dim:", args.embed_dim)
    print("Patch size:", 32 // args.patch_num)
    print("Sequence length:", seqlen)

    model = LitTransformer(num_classes=10, lr=args.lr, epochs=args.max_epochs,
                           depth=args.depth, embed_dim=args.embed_dim, head=args.num_heads,
                           patch_dim=patch_dim, seqlen=seqlen,)

    print(f"Number of model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} million")

    # Use the GPU when available, otherwise CPU.
    accelerator = args.accelerator if torch.cuda.is_available() else 'cpu'
    precision = 32
    print(f"Running on: {accelerator} (precision {precision})")

    # NCCL debug logging: if the 2-rank run ever hangs, the log shows which
    # collective each rank was waiting on.
    os.environ.setdefault("NCCL_DEBUG", "WARN")

    # Keep the checkpoint with the best validation accuracy.
    from pytorch_lightning.callbacks import ModelCheckpoint
    checkpoint_cb = ModelCheckpoint(
        monitor="val_acc", mode="max",
        filename="transformer_cifar10_best", save_top_k=1)

    trainer = Trainer(accelerator=accelerator, devices=1,
                      max_epochs=args.max_epochs, precision=precision,
                      callbacks=[checkpoint_cb],
                      default_root_dir='./lightning_logs')
    trainer.fit(model, datamodule=datamodule)

    # Evaluate the best checkpoint on the held-out test set.
    #
    # NOTE: run the test pass on a single GPU. The test dataloader is not
    # sharded across ranks (see LitCifar10.setup), so a 2-rank DDP test
    # would have one rank finish early and the other block in an
    # all-reduce until the NCCL watchdog aborts the run.
    best_path = checkpoint_cb.best_model_path
    print(f"Best checkpoint: {best_path} (val acc {checkpoint_cb.best_model_score:.4f})")
    best_model = LitTransformer.load_from_checkpoint(
        best_path, num_classes=10, lr=args.lr, epochs=args.max_epochs,
        depth=args.depth, embed_dim=args.embed_dim, head=args.num_heads,
        patch_dim=patch_dim, seqlen=seqlen)
    test_trainer = Trainer(accelerator=accelerator, devices=1,
                           precision=precision, logger=False,
                           default_root_dir='./lightning_logs')
    test_trainer.test(best_model, datamodule=datamodule, verbose=True)


Files already downloaded and verified
Files already downloaded and verified


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


torch.Size([64, 48])
Embed dim: 64
Patch size: 4
Sequence length: 64
Number of model parameters: 0.64 million
Running on: gpu
Files already downloaded and verified
Files already downloaded and verified


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name     | Type               | Params
------------------------------------------------
0 | encoder  | Transformer        | 597 K 
1 | embed    | Linear             | 3.1 K 
2 | fc       | Linear             | 41.0 K
3 | loss     | CrossEntropyLoss   | 0     
4 | accuracy | MulticlassAccuracy | 0     
------------------------------------------------
641 K     Trainable params
0         Non-trainable params
641 K     Total params
2.566     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


### 7.5. Sample images from the checkpoint: ground truth vs prediction

This cell loads the best checkpoint saved during training and shows a **1x4
grid** of random test images. Each panel displays the image with its
**ground-truth (GT)** label and the model's **predicted (Pred)** label.
Titles are green when the prediction matches the ground truth and red when
it does not. Red panels typically expose the classic CIFAR confusions
(e.g. 3/5, 8/9, 0/4), which are the same confusions you see in the
per-class accuracy table and the confusion matrix in section 7.


In [ ]:
import os
import random
import torch
import matplotlib.pyplot as plt
from einops import rearrange

# Find the best checkpoint saved by the ModelCheckpoint callback
ckpt_path = None
for root, dirs, files in os.walk("lightning_logs"):
    for f in files:
        if f == "transformer_cifar10_best.ckpt":
            ckpt_path = os.path.join(root, f)
assert ckpt_path is not None, "No checkpoint found. Run the training cell first."
print(f"Loading checkpoint: {ckpt_path}")

# Load the model from the checkpoint
best_model = LitTransformer.load_from_checkpoint(ckpt_path)
best_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model.to(device)

# Load the test set (raw images, not patches)
test_set = torchvision.datasets.CIFAR10(root="~/data", train=False,
                                        download=True,
                                        transform=torchvision.transforms.ToTensor())

# Pick 4 random test images
random.seed(42)
idxs = random.sample(range(len(test_set)), 4)
images = torch.stack([test_set[i][0] for i in idxs]).to(device)
labels = torch.tensor([test_set[i][1] for i in idxs]).to(device)

# The model expects patches: (B, C, H, W) -> (B, N, patch_dim)
with torch.no_grad():
    x = rearrange(images, 'b c (p1 h) (p2 w) -> b (p1 p2) (c h w)', p1=4, p2=4)
    preds = best_model(x).argmax(dim=1)

# Render the 1x4 grid
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, img, gt, pred in zip(axes, images.cpu(), labels.cpu(), preds.cpu()):
    ax.imshow(img.clamp(0, 1))
    ok = gt == pred
    ax.set_title(f"GT {gt} | Pred {pred}", color="green" if ok else "red",
                 fontsize=11, fontweight="bold")
    ax.axis("off")
plt.suptitle(f"Transformer CIFAR10 checkpoint", fontsize=10)
plt.tight_layout()
plt.show()
